# EDGAR Financial Sentiment — Part 2: Fine-Tuning GPT-2

Adapted from **Assignment 2, §2** (sentiment as next-token prediction), retargeted to finance.

**The question this part answers:** did the continued pretraining from Part 1 actually *help* on a downstream task? We fine-tune two models on the **Financial PhraseBank** sentiment benchmark — **(A)** vanilla `gpt2` and **(B)** your Part-1 EDGAR-pretrained `gpt2` — changing exactly one variable (the starting weights) and compare accuracy.

**What you'll practice (this is the whole point):** writing a `torch.utils.data.Dataset`, batching it with a `DataLoader`, defining an `nn.Module`, and hand-writing the train / test loops. Those four things are blanked out for you in the `_PRACTICE` notebook — fill them in, then check against this answer key.

> **Run in Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).

### The trick: reframe classification as next-token prediction

Instead of bolting a classifier head onto GPT-2, we lean on what a language model already does well — predicting the next word. We wrap each sentence in a prompt so that the *natural next word* is the sentiment label:

> `Here is a sentence from a financial news report: <sentence> ... The sentiment expressed about the company is` **positive**

Then the model's job is just to predict that final word. We score it by comparing the logits of the three sentiment words (`negative`, `neutral`, `positive`) at the last position. This is the same framing you used on IMDB in Assignment 2 §2, now 3-class and financial.

## 0. Setup

In [ ]:
# Pin datasets to the class version: edgar-corpus and the phrasebank loader are script-based
# datasets that newer versions of `datasets` no longer load.
!pip install -q -U transformers
!pip install -q datasets==2.21.0

In [ ]:
import copy, random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, GPT2LMHeadModel
from datasets import load_dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

## 1. Config & seeds

Everything tunable lives here. Seeds match Part 1 so runs are comparable.

In [ ]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

# --- Fine-tuning hyperparameters ---
max_len     = 64      # tokens per prompted example (PhraseBank sentences are short)
batch_size  = 4
lr          = 1e-5
TRAIN_STEPS = 2000    # batches of fine-tuning per model
TEST_STEPS  = 500     # batches to evaluate on

# --- Prompt-as-classifier framing (mirrors Assignment 2 §2) ---
prompt_pre_text  = 'Here is a sentence from a financial news report: '
prompt_post_text = ' ... The sentiment expressed about the company is'

# Financial PhraseBank label ids -> the word we want the model to predict next.
# (0=negative, 1=neutral, 2=positive — confirmed against the dataset below.)
classification_tokenset = {0: 'negative', 1: 'neutral', 2: 'positive'}

## 2. Load GPT-2

Load `gpt2`, add a `[PAD]` token, and resize the embeddings — exactly as in Part 1. This is the **base** model (variant A). We keep the tokenizer's default (right) padding for now; we switch to left-padding later, only for the short classification sentences (Section 5 explains why).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

base_gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
base_gpt2_model.resize_token_embeddings(len(tokenizer))   # account for the added [PAD] token
print('GPT-2 loaded. Vocab size:', len(tokenizer))

## 3. Recreate the Part-1 EDGAR-pretrained model  *(provided — not today's exercise)*

This block is a condensed copy of **Part 1**: it deep-copies the base GPT-2 and continues pretraining it on EDGAR 10-K text, producing `edgar_pretrained_model` (variant B). It is given to you in full because Part 1 already covered this material — today's exercise is the fine-tuning below.

*Tip: if you saved your Part-1 model to Google Drive, you can replace this whole section with a single `GPT2LMHeadModel.from_pretrained(<drive_path>)` to save a few minutes.*

In [ ]:
# --- Condensed Part 1: continued pretraining on EDGAR 10-K text (uses default right-padding) ---
EDGAR_YEAR, TARGET_CHUNKS, WORDS_PER_CHUNK = 'year_2020', 15000, 120
PRETRAIN_STEPS, PRETRAIN_MAXLEN = 1000, 100

def _chunk(text, n=WORDS_PER_CHUNK):
    w = text.split()
    return [' '.join(w[i:i+n]) for i in range(0, len(w), n)]

print('Streaming EDGAR 10-K filings for continued pretraining...')
_stream = load_dataset('eloukas/edgar-corpus', EDGAR_YEAR, split='train',
                       streaming=True, trust_remote_code=True)
_chunks = []
for _rec in _stream:
    _secs = [v for k, v in _rec.items() if k.startswith('section_') and isinstance(v, str) and v.strip()]
    _chunks.extend(_chunk('\n'.join(_secs)))
    if len(_chunks) >= TARGET_CHUNKS:
        break
random.shuffle(_chunks); _chunks = _chunks[:TARGET_CHUNKS]

class _PretrainData(Dataset):
    def __init__(self, texts):
        enc = tokenizer(texts, max_length=PRETRAIN_MAXLEN, truncation=True,
                        padding='max_length', return_tensors='pt')
        keep = enc['attention_mask'][:, PRETRAIN_MAXLEN - 1] > 0   # keep only full-length chunks
        self.data = enc['input_ids'][keep].to(device)
    def __len__(self): return self.data.shape[0]
    def __getitem__(self, i):
        return {'input': self.data[i][:-1], 'labels': self.data[i][1:]}

_loader = DataLoader(_PretrainData(_chunks), batch_size=4, shuffle=True, drop_last=True)

edgar_pretrained_model = copy.deepcopy(base_gpt2_model)
_opt = torch.optim.AdamW(edgar_pretrained_model.parameters(), lr=1e-5)
_ce  = nn.CrossEntropyLoss()
edgar_pretrained_model.train()
print(f'Continued pretraining for {PRETRAIN_STEPS} steps...')
for _step, _b in enumerate(_loader):
    if _step >= PRETRAIN_STEPS: break
    _logits = edgar_pretrained_model(_b['input']).logits.reshape(-1, len(tokenizer))
    _opt.zero_grad()
    _loss = _ce(_logits, _b['labels'].reshape(-1))
    _loss.backward(); _opt.step()
    if (_step + 1) % 200 == 0:
        print(f'  step {_step+1:>4}: loss {_loss.item():.3f}')
print('Done. edgar_pretrained_model is ready.')

## 4. Load the Financial PhraseBank benchmark

`takala/financial_phrasebank`, config `sentences_allagree` (~2,264 sentences all annotators agreed on; 3 classes). There's no predefined split, so we make our own **stratified** 80/20 split (stratifying keeps the class balance the same in train and test).

In [ ]:
pb = load_dataset('takala/financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
pb = pb['train'].train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']

label_names = pb_train.features['label'].names   # ['negative', 'neutral', 'positive']
print('Label index -> name:', dict(enumerate(label_names)))
print('Train size:', len(pb_train), '| Test size:', len(pb_test))
print('Example row:', pb_train[0])

## 5. Build the classification `Dataset`  &larr; **your code**

**Padding note (important — this is new vs. Part 1).** PhraseBank sentences are short, so we must pad them to a common `max_len` to batch them. Because we read the sentiment prediction off the **last position** of a causal model, that last position has to be a *real* token. So we:
- **pad on the left** (`padding_side='left'`) — pad tokens go at the front, real tokens end at the last position;
- **truncate on the left** (`truncation_side='left'`) — if something is too long, trim the *front* so the `... is` cue at the end always survives;
- carry an **`attention_mask`** so the model ignores the pad tokens.

For each row you build `pre_text + sentence + post_text` and store: the padded `input_ids`, the `attention_mask`, the **token id** of the sentiment word (the next-token target, for the loss), and the **class index** 0/1/2 (for accuracy).

In [ ]:
tokenizer.padding_side    = 'left'   # last position = a real token (see note above)
tokenizer.truncation_side = 'left'   # if too long, trim the front, keep the post-prompt cue

In [ ]:
class ClassificationData(Dataset):
    """Turn (sentence, label) rows into prompted next-token-prediction examples.

    For each row, build the prompt  pre_text + sentence + post_text , left-pad it to
    max_len, and store a dict with:
      'input_ids'      : padded token ids                          (max_len,)
      'attention_mask' : 1 for real tokens, 0 for left-pad tokens  (max_len,)
      'label'          : token id of the sentiment word to predict (scalar, int64)
      'label_idx'      : the class index 0/1/2                      (scalar, int64)
    """
    def __init__(self, hf_split, tokenizer, max_len,
                 prompt_pre_text, prompt_post_text, classification_tokenset):
        self.data = []
        # the single token id for each sentiment word (leading space matters for GPT-2 BPE)
        token_id_for_idx = {idx: tokenizer.encode(' ' + word)[0]
                            for idx, word in classification_tokenset.items()}

        for ex in hf_split:
            ### YOUR CODE HERE
            prompt = prompt_pre_text + ex['sentence'] + prompt_post_text
            enc = tokenizer(prompt, max_length=max_len, truncation=True,
                            padding='max_length', return_tensors='pt')
            self.data.append({
                'input_ids':      enc['input_ids'].squeeze(0).to(device),
                'attention_mask': enc['attention_mask'].squeeze(0).to(device),
                'label':     torch.tensor(token_id_for_idx[ex['label']], dtype=torch.int64, device=device),
                'label_idx': torch.tensor(ex['label'], dtype=torch.int64, device=device),
            })
            ### END YOUR CODE

    def __len__(self):
        ### YOUR CODE HERE
        return len(self.data)
        ### END YOUR CODE

    def __getitem__(self, index):
        ### YOUR CODE HERE
        return self.data[index]
        ### END YOUR CODE

### Sanity-check your `Dataset`
Run this after you fill in the cell above — it verifies shapes, the left-padding, and that the three sentiment words map to distinct token ids.

In [ ]:
_probe = ClassificationData(pb_train, tokenizer, max_len, prompt_pre_text, prompt_post_text, classification_tokenset)
print('Examples built:', len(_probe))

_ids = {w: tokenizer.encode(' ' + w)[0] for w in classification_tokenset.values()}
print('Contrast token ids:', _ids)
assert len(set(_ids.values())) == 3, 'Sentiment words collide on their first BPE token — pick different words.'

_ex = _probe[0]
print('input_ids shape:', tuple(_ex['input_ids'].shape), '| real tokens:', int(_ex['attention_mask'].sum()))
print('label token id:', int(_ex['label']), '->', repr(tokenizer.decode([int(_ex['label'])])),
      '| label_idx:', int(_ex['label_idx']))
print('Decoded tail:', repr(tokenizer.decode(_ex['input_ids'][-40:])))

## 6. Define the model wrapper `nn.Module`  &larr; **your code**

A thin wrapper around a GPT-2 language model (just like `TokenPredictionNetworkClass` in Assignment 2, plus an attention mask). `forward` receives a **batch dict**, runs the LM on `input_ids` *with* the `attention_mask`, and returns only the **last-position** logits — shape `(batch, vocab)`.

In [ ]:
class TokenPredictionNetwork(nn.Module):
    def __init__(self, language_model):
        ### YOUR CODE HERE
        super().__init__()
        self.language_model = language_model
        ### END YOUR CODE

    def forward(self, batch):
        ### YOUR CODE HERE
        out = self.language_model(batch['input_ids'], attention_mask=batch['attention_mask'])
        last_token_logits = out.logits[:, -1, :]     # (batch, vocab)
        return last_token_logits
        ### END YOUR CODE

loss_fn = nn.CrossEntropyLoss()

## 7. DataLoaders

Wrap train and test sets. We also precompute the three contrast token ids in a fixed (neg, neu, pos) order for the test loop's class-accuracy.

In [ ]:
train_data = ClassificationData(pb_train, tokenizer, max_len, prompt_pre_text, prompt_post_text, classification_tokenset)
test_data  = ClassificationData(pb_test,  tokenizer, max_len, prompt_pre_text, prompt_post_text, classification_tokenset)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False, drop_last=True)
print('Train batches:', len(train_loader), '| Test batches:', len(test_loader))

contrast_token_ids = [tokenizer.encode(' ' + classification_tokenset[i])[0] for i in range(3)]
print('contrast_token_ids (neg, neu, pos):', contrast_token_ids)

## 8. Train and test loops  &larr; **your code**

Hand-written, same shape as Assignment 2. The **train loop** is plain next-token fine-tuning against the sentiment token. The **test loop** reports two accuracies:
- **class accuracy** — of the three sentiment-word logits, is the right one the biggest?
- **token accuracy** — is the sentiment word the model's overall top next-token (out of the whole vocab)?

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, reporting_interval=200, steps=None):
    """Fine-tune for sentiment via next-token prediction.

    - put the model in train mode; track a running loss and a batch count
    - for each batch (stop once you've done `steps` batches):
        - model(batch) gives the last-position logits   -> (batch, vocab)
        - the targets are batch['label']                -> (batch,)
        - zero_grad, CrossEntropyLoss, backward, optimizer.step()
        - accumulate the loss
    - print the running average loss every `reporting_interval` batches
    """
    ### YOUR CODE HERE
    model.train()
    running, n = 0.0, 0
    for step, batch in enumerate(dataloader):
        if steps is not None and step >= steps:
            break
        logits = model(batch)
        targets = batch['label']
        optimizer.zero_grad()
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()
        running += loss.item(); n += 1
        if (step + 1) % reporting_interval == 0:
            print(f'  step {step+1:>4}: avg train loss {running/n:.4f}')
    print(f'  done: avg train loss {running/max(n,1):.4f}')
    ### END YOUR CODE

In [ ]:
def test_loop(dataloader, model, loss_fn, contrast_token_ids, reporting_interval=200, steps=None):
    """Evaluate: class accuracy and token accuracy.

    - put the model in eval mode; wrap the loop in torch.no_grad()
    - for each batch (stop after `steps`):
        - logits = model(batch)   -> (batch, vocab); add the loss vs batch['label']
        - CLASS accuracy:  logits[:, contrast]  -> (batch, 3); argmax over the 3 = predicted
          class index 0/1/2; compare to batch['label_idx']
        - TOKEN accuracy:  argmax over the full vocab; compare to batch['label']
    - report token acc, class acc and avg loss; return the final class accuracy
    """
    ### YOUR CODE HERE
    model.eval()
    test_loss, correct_class, correct_token, total, n = 0.0, 0, 0, 0, 0
    contrast = torch.tensor(contrast_token_ids, device=device)
    with torch.no_grad():
        for step, batch in enumerate(dataloader):
            if steps is not None and step >= steps:
                break
            logits = model(batch)
            test_loss += loss_fn(logits, batch['label']).item(); n += 1
            total += batch['label'].shape[0]

            class_logits = logits[:, contrast]                 # (batch, 3)
            pred_class = torch.argmax(class_logits, dim=-1)
            correct_class += torch.sum(pred_class == batch['label_idx']).item()

            pred_token = torch.argmax(logits, dim=-1)
            correct_token += torch.sum(pred_token == batch['label']).item()

            if (step + 1) % reporting_interval == 0:
                print(f'  step {step+1:>4}: class acc {100*correct_class/total:.1f}% | '
                      f'token acc {100*correct_token/total:.1f}% | loss {test_loss/n:.4f}')
    class_acc = 100*correct_class/max(total,1)
    token_acc = 100*correct_token/max(total,1)
    print(f'  FINAL: class acc {class_acc:.1f}% | token acc {token_acc:.1f}% | avg loss {test_loss/max(n,1):.4f}')
    return class_acc
    ### END YOUR CODE

## 9. Run the comparison — base vs. EDGAR-pretrained

The exact same fine-tuning recipe is applied to both starting models; only the starting weights differ (our one variable). We `deepcopy` each model so the two runs stay independent.

In [ ]:
def finetune_and_eval(language_model, name):
    print(f'\n=== {name} ===')
    net   = TokenPredictionNetwork(copy.deepcopy(language_model)).to(device)
    optim = torch.optim.AdamW(net.parameters(), lr=lr)
    print('Fine-tuning...')
    train_loop(train_loader, net, loss_fn, optim, steps=TRAIN_STEPS)
    print('Evaluating...')
    return test_loop(test_loader, net, loss_fn, contrast_token_ids, steps=TEST_STEPS)

acc_base  = finetune_and_eval(base_gpt2_model,        'A: vanilla GPT-2')
acc_edgar = finetune_and_eval(edgar_pretrained_model, 'B: EDGAR-pretrained GPT-2')

print('\n================ RESULTS ================')
print(f'A  vanilla GPT-2           class accuracy: {acc_base:.1f}%')
print(f'B  EDGAR-pretrained GPT-2  class accuracy: {acc_edgar:.1f}%')
print(f'Delta (continued-pretraining payoff):      {acc_edgar - acc_base:+.1f} pts')

## 10. What to look for

- **B &ge; A** is the clean story: continued pretraining on EDGAR gave the model a head start on financial language that survives fine-tuning. That's the same logic that makes **FinBERT** beat vanilla BERT — and Part 3 tests exactly that with the published models.
- **A &asymp; B, or even B &lt; A, is a real result too — not a failure.** Plausible reasons worth writing down: only 1,000 pretraining steps on a single year of 10-Ks; 10-K boilerplate is stylistically far from PhraseBank's news-headline sentences; fine-tuning can wash out a small pretraining edge; or the metric is just noisy on ~450 test sentences.
- **Change-one-variable discipline:** A and B share *everything* (data, prompt, loss, steps, lr, seed) except the starting weights — so any gap is attributable to the EDGAR pretraining.

**Things to try once it runs:** tweak `prompt_pre_text` / `prompt_post_text` (the framing strongly affects accuracy, as you saw on IMDB); try dropping `neutral` for a binary task; raise `TRAIN_STEPS`.

**Next (Part 3):** swap this whole approach for a BERT `[CLS]` classifier and compare `bert-base-cased` vs `ProsusAI/finbert` — the published, fully-trained version of the Part-1/Part-2 idea.